# E4 · Concellos de la provincia de A Coruña

**Sistemas de Big Data · UD1 · Tema 3** — Ejercicio individual

Extraemos de Wikipedia la población de los 93 concellos de la provincia de A Coruña, la validamos con la propia página y respondemos cuatro preguntas sobre cómo se reparte la población.

Fuente: [List of municipalities in A Coruña](https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a) (Wikipedia en inglés).

In [1]:
import re
import time
from io import StringIO
from pathlib import Path
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_municipalities_in_A_Coru%C3%B1a"
CABECERAS = {"User-Agent": "curso-sbd-e4-concellos/1.0 (raul.antelobdi@gmail.com)"}

RAW = Path("../datos/raw")
CLEAN = Path("../datos/clean")
RAW.mkdir(parents=True, exist_ok=True)
CLEAN.mkdir(parents=True, exist_ok=True)
FICHERO = RAW / "concellos_coruna.html"

## 4.1 Antes de descargar

### robots.txt

In [ ]:
# RobotFileParser.read() usa el User-Agent por defecto de urllib, que Wikipedia bloquea (403),
# y entonces interpreta que todo está prohibido. Descargamos robots.txt con requests y nuestro
# User-Agent, y se lo pasamos al parser.
r_robots = requests.get("https://en.wikipedia.org/robots.txt", headers=CABECERAS, timeout=10)
print("Estado robots.txt:", r_robots.status_code)
robots = RobotFileParser()
robots.parse(r_robots.text.splitlines())
print("¿Permitido descargar la página?", robots.can_fetch(CABECERAS["User-Agent"], URL))
print("Crawl-delay:", robots.crawl_delay(CABECERAS["User-Agent"]))

Estado robots.txt: 200
¿Permitido descargar la página? True
Crawl-delay: None


### Licencia y datos personales

- **Licencia.** Los textos de Wikipedia se publican bajo **CC BY-SA 4.0**. Permite copiar, redistribuir y adaptar el contenido (también para usos comerciales) con dos condiciones: **atribuir** la fuente (indicar la página de Wikipedia y la licencia) y **compartir igual**, es decir, distribuir los datos derivados bajo la misma licencia. Por eso el CSV final y este notebook citan la fuente y la licencia.
- **Datos personales.** La página no contiene datos personales: solo nombres de concellos (entidades administrativas) y cifras de población **agregadas** por concello. No hay nombres de personas, direcciones ni ningún dato que permita identificar a una persona concreta.

## 4.2 Descarga

Guardamos el HTML **en bytes y sin modificar**. Si el fichero ya existe, no se vuelve a descargar.

In [3]:
def descargar(url, destino):
    if destino.exists():
        return "ya existía, no se descarga"
    r = requests.get(url, headers=CABECERAS, timeout=10)
    r.raise_for_status()
    destino.write_bytes(r.content)
    time.sleep(1)
    return f"descargado ({len(r.content):,} bytes)"

print(descargar(URL, FICHERO))

ya existía, no se descarga


## 4.3 Extracción

Inspeccionando la página (F12) se ve que la lista está en una `<table class="wikitable sortable">` con dos columnas: `Name` y `Population (2024)[20]`. No la buscamos por su posición, sino por el contenido de la cabecera: la tabla que tiene una cabecera que empieza por «Population». Usamos bytes para que BeautifulSoup lea el `<meta charset>`.

In [4]:
sopa = BeautifulSoup(FICHERO.read_bytes(), "html.parser")

def encontrar_tabla(sopa):
    for tabla in sopa.find_all("table"):
        cabeceras = [th.get_text(strip=True) for th in tabla.find_all("th")]
        if any(h.startswith("Population") for h in cabeceras) and "Name" in cabeceras:
            return tabla
    raise ValueError("No se encontró la tabla de población")

tabla = encontrar_tabla(sopa)
filas = []
for tr in tabla.find_all("tr"):
    celdas = tr.find_all("td")
    if len(celdas) == 2:                      # salta la fila de cabeceras (solo tiene th)
        filas.append({"concello": celdas[0].get_text(strip=True),
                      "poblacion_texto": celdas[1].get_text(strip=True)})

concellos = pd.DataFrame(filas)
print(concellos.shape)
concellos.head()

(93, 2)


,concello,poblacion_texto
0,Abegondo,"5,578"
1,Ames,"32,812"
2,Aranga,"1,799"
3,Ares,"6,156"
4,Arteixo,"34,038"


## 4.4 Limpieza

Antes de convertir, comprobamos que todos los valores tienen el formato esperado: dígitos con `,` como separador de miles (`1,234`).

In [5]:
formato_ok = concellos["poblacion_texto"].str.fullmatch(r"\d{1,3}(,\d{3})*")
print("Valores con el formato esperado:", formato_ok.sum(), "de", len(concellos))
concellos[~formato_ok]

Valores con el formato esperado: 92 de 93


,concello,poblacion_texto
9,Boimorto,1.801


Hay **un valor que no sigue el formato**: Boimorto aparece como `1.801`, con **punto** en vez de coma como separador de miles (es la convención española). Si lo convirtiéramos como el resto (quitando solo las comas) o con `float`, daría `1.801` habitantes, un error de tres órdenes de magnitud. Como el resto de la columna usa la coma como separador de miles y un concello no puede tener 1,8 habitantes, interpretamos el punto como separador de miles y lo quitamos. Lo verificaremos con la suma total en la validación.

In [6]:
concellos["poblacion"] = (concellos["poblacion_texto"]
                          .str.replace(",", "", regex=False)
                          .str.replace(".", "", regex=False)
                          .astype(int))
concellos.loc[~formato_ok]

,concello,poblacion_texto,poblacion
9,Boimorto,1.801,1801


## 4.5 Validación

In [7]:
print("Concellos:", len(concellos), "| únicos:", concellos["concello"].nunique())
assert len(concellos) == 93 and concellos["concello"].is_unique

Concellos: 93 | únicos: 93


Extraemos del texto de la página la población total de la provincia (está en la introducción: «…with 1,128,320 inhabitants») y la comparamos con la suma de la tabla.

In [8]:
texto = sopa.get_text(" ", strip=True)
total_pag = int(re.search(r"with ([\d,]+) inhabitants", texto).group(1).replace(",", ""))
suma = concellos["poblacion"].sum()
print("Total de la página:", f"{total_pag:,}")
print("Suma de la tabla  :", f"{suma:,}")
print("¿Coinciden?", total_pag == suma)

Total de la página: 1,128,320
Suma de la tabla  : 1,128,320
¿Coinciden? True


Ahora leemos la misma tabla con `pd.read_html` (sin indicarle `thousands`) y comparamos con nuestro resultado.

In [9]:
auto = pd.read_html(StringIO(FICHERO.read_text(encoding="utf-8")), match="Population")[0]
auto.columns = ["concello", "poblacion_auto"]
print(auto.dtypes, "\n")
cmp = concellos.merge(auto, on="concello")
dif = cmp[cmp["poblacion"] != cmp["poblacion_auto"]]
print("Diferencias:", len(dif))
dif

concello              str
poblacion_auto    float64
dtype: object 

Diferencias: 1


,concello,poblacion_texto,poblacion,poblacion_auto
9,Boimorto,1.801,1801,1.801


`read_html` detecta la columna como decimal (`float64`) porque «1.801» se parece a un número decimal. Resultado: Boimorto queda con **1,801 habitantes** en vez de 1801, y la suma total no coincide con el total de la página. Con `thousands=","` no se arreglaría, porque ese valor usa punto. Otra diferencia menor es el tipo (`float` frente a `int`). Nuestra limpieza explícita detecta el caso, aplica una corrección justificada y se valida con el total de la página.

## 4.6 Preguntas

In [10]:
datos = concellos.sort_values("poblacion", ascending=False).reset_index(drop=True)
total = datos["poblacion"].sum()

# 1. Cinco concellos más poblados
datos.head(5)[["concello", "poblacion"]]

,concello,poblacion
0,A Coruña,249261
1,Santiago de Compostela,99536
2,Ferrol,64218
3,Narón,39285
4,Oleiros,38333


**1.** Los cinco más poblados son A Coruña (249.261), Santiago de Compostela, Ferrol, Narón y Oleiros; A Coruña sola tiene más del doble que Santiago, el segundo.

In [11]:
# 2. Concellos con menos de 5.000 habitantes
n_pequenos = (datos["poblacion"] < 5000).sum()
print(f"{n_pequenos} concellos ({n_pequenos / len(datos):.1%})")

43 concellos (46.2%)


**2.** 43 de los 93 concellos (46,2 %), casi la mitad, tienen menos de 5.000 habitantes: la provincia tiene muchos concellos pequeños, que juntos suman poca población.

In [12]:
# 3. Porcentaje de la población en los tres más poblados
top3 = datos.head(3)["poblacion"].sum()
print(f"{top3:,} habitantes → {top3 / total:.1%} de la provincia")

413,015 habitantes → 36.6% de la provincia


**3.** Tres concellos (A Coruña, Santiago y Ferrol) reúnen el 36,6 % de la población provincial: un 3 % de los concellos concentra más de un tercio de los habitantes.

In [13]:
# 4. Concellos necesarios para llegar a la mitad
datos["acumulada"] = datos["poblacion"].cumsum() / total
n_mitad = int((datos["acumulada"] < 0.5).sum() + 1)
print(f"Hacen falta {n_mitad} concellos para reunir la mitad de la población")
datos.head(n_mitad)[["concello", "poblacion", "acumulada"]].round(3)

Hacen falta 8 concellos para reunir la mitad de la población


,concello,poblacion,acumulada
0,A Coruña,249261,0.221
1,Santiago de Compostela,99536,0.309
2,Ferrol,64218,0.366
3,Narón,39285,0.401
4,Oleiros,38333,0.435
5,Arteixo,34038,0.465
6,Ames,32812,0.494
7,Carballo,31595,0.522


**4.** Hacen falta solo 8 concellos de 93 (el 8,6 %) para superar el 50 % de la población (52,2 % con el octavo); los otros 85 reparten la otra mitad.

## 4.7 Guardar

Datos derivados de Wikipedia (CC BY-SA 4.0), por lo que se distribuyen bajo la misma licencia.

In [14]:
(concellos[["concello", "poblacion"]]
    .sort_values("concello")
    .to_csv(CLEAN / "concellos_coruna.csv", index=False, encoding="utf-8"))
print(pd.read_csv(CLEAN / "concellos_coruna.csv").shape)

(93, 2)
